# Silver: ERP Customers
**Source:** databricks_bootcamp_dwb.bronze.erp_cust_az12  
**Target:** databricks_bootcamp_dwb.silver.erp_customers

ERP = 'Enterprise Resource Planning'

# Read Data From Bronze Layer

In [0]:
# imports
from functools import reduce
from operator import add

from pyspark.sql.functions import (
    col, count, countDistinct, expr, isnull,
    monotonically_increasing_id, row_number, sum, trim, when
)
from pyspark.sql.types import DateType, DecimalType, IntegerType, StringType
from pyspark.sql.window import Window

In [0]:
df = spark.table("databricks_bootcamp_dwb.bronze.erp_cust_az12")
df.display()

# EDA

In [0]:
# Check for nulls in each column
df.select([count(when(isnull(c), c)).alias(c) for c in df.columns]).display()

In [0]:
# Check for duplicates in each column
print(f"total rows: {df.count()}")
for column in df.columns:
    duplicate_count = df.groupBy(column).count().filter(col("count") > 1).agg(count("*")).collect()[0][0]
    print(f"{column}: {duplicate_count} duplicate values")

In [0]:
# Get distinct GEN values from the table
df.select("GEN").distinct().display() 

In [0]:
# Check data types of all columns
df.printSchema()

# Data Transformations

In [0]:
# trim strings
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

display(df)

df.select("GEN").distinct().display() 

In [0]:
# Map GEN column values
df = df.withColumn(
    "GEN",
    when((col("GEN").isNull()) | (col("GEN") == ""), "n/a")
    .when(col("GEN") == "M", "Male")
    .when(col("GEN") == "F", "Female")
    .otherwise(col("GEN"))
)

df.select("GEN").distinct().display()

In [0]:
# make friendly column names
RENAME_MAP = {
    "BDATE": "birth_date",
    "GEN": "gender",
}

df = df.select([col(c).alias(RENAME_MAP.get(c, c)) for c in df.columns])
display(df)

In [0]:
# Schema transformation for join compatibility
# Justification: see table_relationship_exploration notebook and silver_crm_prd_info for pattern
# Extract customer_id (last 5 chars) and category_id (everything except last 5 chars) from CID
# Then drop CID column to avoid duplication

from pyspark.sql.functions import substring, length

# Extract customer_id from CID (last 5 characters)
df = df.withColumn(
    "customer_id",
    substring(col("CID"), -5, 5)
)

# Extract category_id from CID (everything except last 5 characters)
df = df.withColumn(
    "category_id",
    substring(col("CID"), 1, length(col("CID")) - 5)
)

# Drop CID column
df = df.drop("CID")

print("Schema after transformation:")
df.printSchema()
display(df)

In [0]:
# Define target data types for non-string columns.
type_mappings = {
    "birth_date": DateType(),
}

# Apply type casting
for field in df.schema.fields:
    column_name = field.name
    
    if column_name in type_mappings:
        target_type = type_mappings[column_name]
        if target_type is not None:
            df = df.withColumn(column_name, col(column_name).cast(target_type))
    else:
        # All other columns should be StringType
        df = df.withColumn(column_name, col(column_name).cast(StringType()))

print("Data types after enforcement:")
df.printSchema()

In [0]:
from pyspark.sql.functions import current_timestamp

# Add timestamp column to track when row was written to silver table
df = df.withColumn("silver_updated_at", current_timestamp())

print(f"Added silver_updated_at column")
display(df)

# Write To Silver Table

In [0]:
df.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable("databricks_bootcamp_dwb.silver.erp_customers")

In [0]:
%sql
SELECT * FROM databricks_bootcamp_dwb.silver.erp_customers
LIMIT 100